## inference for means 

In [1]:
# # _rd_00.py 
# 평균 추론. muhat, se, 신뢰구간, 검정통계량, pvalue.  
# 두 평균의 비교, 별도 진행 중.
# raw data - 연봉, 경력 기간, 성별.
# _rd_00.py ==> 이건 벤치마크용.


import os
import numpy as np                          # numpy 라이브러리 전체. 
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네. 
import scipy as sci 
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis 
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import DescrStatsW, ztest
from scipy.stats import ttest_1samp


### 자료셋 정리
gender, height, 성별 키 

In [2]:
# 1. 데이터 준비. 읽기. 생성. 
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat = pd.read_csv(dat_url) 
df_dat.head() 

gn = df_dat['gender'].to_numpy()   # 이렇게 하면 배열로 전환. dataframe -> NumPy 배열
hgt = df_dat['ht'].to_numpy()
gndr = (gn == 1).astype(int)    # gn = 1, 2; gndr = 1, 0. 1/0으로 전환. 

df_dat['gndr'] = gndr               #   이것은 위에 1/0 자료를 추가. 성별임.

hgt_m = hgt[gndr == 1]  # 성별 1, m 그룹 키. 그룹별 자료 분리, 배열은 hgt 하나임
hgt_f = hgt[gndr == 0]  #     0, f. 키 그룹별 자료 분리

n_all = len(df_dat) 
n_m = len(hgt_m)
n_f = len(hgt_f)



\begin{align}
 \hat \mu & \sim N \left(\mu, SE^2 \right) \\
 \widehat{SE} & = \sqrt{ \sigma^2 \over n } 
\end{align}

$ 100(1-\alpha) \% $ 신뢰구간 
\begin{align}  
\hat \mu  & \pm z_{\alpha/2} \times \widehat{ SE }  \\  
\hat \mu  & \pm t_{\alpha/2, df} \times \widehat{ SE }  
\end{align} 


In [25]:

# 분포 임계치, 양방향, 우측값. 
alpha = 0.05                          # significance level, two side.
zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side
#    tcv_r = stats.t.ppf(1 - alpha / 2, df= n_tall)

# 3. 평균 키 추정치: 전체, 성별, 표준오차
# 추정치: 평균, 표준오차, overall
# overall 
muhat = hgt.mean()
se_muhat = hgt.std(ddof=1) / np.sqrt( n_all )

# 그룹별( m - f)
muhat_m = hgt_m.mean()
muhat_f = hgt_f.mean()

se_muhat_m = hgt_m.std(ddof=1)/ np.sqrt( n_m )
se_muhat_f = hgt_f.std(ddof=1) / np.sqrt( n_f )

# 신뢰구간, right, left, 
# overall 
ci_r = muhat + zcv_r * se_muhat 
ci_l = muhat - zcv_r * se_muhat  


# 그룹별 ( m - f )
ci_rm = muhat_m + zcv_r * se_muhat_m 
ci_lm = muhat_m - zcv_r * se_muhat_m 

ci_rf = muhat_f + zcv_r * se_muhat_f 
ci_lf = muhat_f - zcv_r * se_muhat_f 

print( "            muhat,    se of muhat,     confidence interval ")
print(" overall ", muhat, se_muhat, ci_l, ci_r ) 
print("  female ", muhat_f, se_muhat_f, ci_lf, ci_rf ) 
print( "  male  ", muhat_m, se_muhat_m, ci_lm, ci_rm ) 


            muhat,    se of muhat,     confidence interval 
 overall  162.73498112082672 0.06068057044374317 162.61604938819565 162.85391285345779
  female  157.202842331069 0.05396534102673514 157.09707220624318 157.30861245589483
  male   169.65819910514543 0.06677075550074704 169.52733082914344 169.78906738114742


\begin{align}
 H_0 & : \mu  = \mu_0  \\
 H_A & : \mu  \ne 0  
\end{align}

$$
 T_0 = { \hat\mu - \mu_0 \over \widehat{ SE } }
$$

$$
 \text{reject } H_0  \\
 \text{ if } |T_0| > z_{\alpha/2} \\ 
 \text{ if  pvalue } < \alpha
$$

In [30]:
# 가설검정. 전체, 톨 
# 귀무가설 H0: mu = mu0
# 검정통계치, pvalue 

mu_zero = 160

t_0 = np.abs( ( muhat - mu_zero ) / se_muhat )            # overall 
t_0m = np.abs( ( muhat_m - mu_zero ) / se_muhat_m )       # male
t_0f = np.abs( ( muhat_f - mu_zero ) / se_muhat_f )       # female

# overall, whole, total 
yn_h0 = " 'reject h0' " if t_0 > zcv_r else " 'fail to reject h0' "   # 이건 되는 군. 
pval = 2*( 1 - stats.norm.cdf( t_0 ) )
print(" 전체    :  ", t_0 , zcv_r, yn_h0, pval) 

# ***** 여기는 gender = 1, male 
yn_h0m = " 'reject h0' " if t_0m > zcv_r else " 'fail to reject h0' "   # 이건 되는 군. 
pvalm = 2*( 1 - stats.norm.cdf( t_0m ) )
print(" male   : ", t_0m , zcv_r, yn_h0m, pvalm) 

# ***** 여기는 gender = 0, female 
yn_h0f = " 'reject h0' " if t_0f > zcv_r else " 'fail to reject h0' "   # 이건 되는 군. 
pvalf = 2*( 1 - stats.norm.cdf( t_0f ) )
print(" female :  ", t_0f , zcv_r, yn_h0f, pvalf) 


 전체    :   45.07177669600704 1.959963984540054  'reject h0'  0.0
 male   :  144.64714428815128 1.959963984540054  'reject h0'  0.0
 female :   51.83248388155731 1.959963984540054  'reject h0'  0.0


In [44]:


# # 모듈 이용 statsmodels.stats.weightstats

# overall
# # 신뢰구간 
#    from statsmodels.stats.weightstats import DescrStatsW
ds_hgt = DescrStatsW( hgt )               # 모듈이 돌려준 객체 
ci_a = ds_hgt.tconfint_mean(alpha=0.05)   # 이건 tuple 

print("H0: mu =", mu_zero )
print("overall ")
print(f" 표본평균 : {ds_hgt.mean:.4f}")
print(f" 표준편차 : {ds_hgt.std:.4f}")
print(f" 표준오차 : {(ds_hgt.std/np.sqrt(n_all)):.4f}")
print(" 신뢰구간 :", ", ".join(f"{x:.4f}" for x in ci_a ))

# 평균 t 검정. 모듈이 여럿임.
t0_a, pval_a, df_a = ds_hgt.ttest_mean(mu_zero) # mu_zero겠지? 

print(" 검정통계치 ",  t0_a ) #, pval_a  ) #, df_a 
print(" p-value  ",  pval_a  ) #, df_a 

# Male 
ds_hgt_m = DescrStatsW( hgt_m )               # 모듈이 돌려준 객체 
ci_m = ds_hgt_m.tconfint_mean(alpha=0.05)   # 이건 tuple 

print("Male ")
print(f" 표본평균 : {ds_hgt_m.mean:.4f}")
print(f" 표준편차 : {ds_hgt_m.std:.4f}")
print(f" 표준오차 : {(ds_hgt_m.std/np.sqrt(n_m)):.4f}")
print(" 신뢰구간 :", ", ".join(f"{x:.4f}" for x in ci_m ))

t0_m, pval_m, df_m = ds_hgt_m.ttest_mean(mu_zero) # mu_zero겠지? 

print(" 검정통계치 ",  t0_m ) #, pval_a  ) #, df_a 
print(" p-value  ",  pval_m  ) #, df_a 

# female 
ds_hgt_f = DescrStatsW( hgt_f )               # 모듈이 돌려준 객체 
ci_f = ds_hgt_f.tconfint_mean(alpha=0.05)   # 이건 tuple 

print("Female ")
print(f" 표본평균 : {ds_hgt_f.mean:.4f}")
print(f" 표준편차 : {ds_hgt_f.std:.4f}")
print(f" 표준오차 : {(ds_hgt_f.std/np.sqrt(n_f)):.4f}")
print(" 신뢰구간 :", ", ".join(f"{x:.4f}" for x in ci_f ))

t0_f, pval_f, df_f = ds_hgt_f.ttest_mean(mu_zero) # mu_zero겠지? 

print(" 검정통계치 ",  t0_f ) #, pval_a  ) #, df_a 
print(" p-value  ",  pval_f  ) #, df_a 


H0: mu = 160
overall 
 표본평균 : 162.7350
 표준편차 : 8.6087
 표준오차 : 0.0607
 신뢰구간 : 162.6160, 162.8539
 검정통계치  45.07177669600703
 p-value   0.0
Male 
 표본평균 : 169.6582
 표준편차 : 6.3129
 표준오차 : 0.0668
 신뢰구간 : 169.5273, 169.7891
 검정통계치  144.64714428815165
 p-value   0.0
Female 
 표본평균 : 157.2028
 표준편차 : 5.7078
 표준오차 : 0.0540
 신뢰구간 : 157.0971, 157.3086
 검정통계치  -51.832483881557835
 p-value   0.0


### 2개 그룹 평균 비교는 별도 주제로